In [1]:
import sys
from pathlib import Path
import os
import json

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

from Utils.DataUtils import build_ae_dataloaders
from Models.AutoEncoder import AutoEncoder
from Utils.FeatureUtils import extract_features

from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score, fbeta_score, recall_score, precision_score, roc_auc_score, average_precision_score, accuracy_score

In [2]:
# Config
BATCH_SIZE = 256
LR = 1e-3
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# AE training, train only non-fraud
train_loader_ae, val_loader, test_loader, input_dim = build_ae_dataloaders(
    batch_size=BATCH_SIZE,
    mode="ae",
    train_only_nonfraud=True,
    return_labels=True
)

# Train fraud + non-fraud
train_loader_full, _, _, _ = build_ae_dataloaders(
    batch_size=BATCH_SIZE,
    mode="ae",
    train_only_nonfraud=False,
    return_labels=True
)

print("Input dim:", input_dim)

[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

[INFO] Train: (455902, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
[INFO] Project root: C:\DeepLearning\DL_project
[INFO] Data dir: C:\DeepLearning\DL_project\data
[INFO] Loading train data from: C:\DeepLearning\DL_project\data\train_transaction.csv
[INFO] Loading test data from: C:\DeepLearning\DL_project\data\test_transaction.csv
[INFO] Train shape: (590540, 394)
[INFO] Test shape: (506691, 393)


C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col + "_missing"] = df[col].isna().astype(int)
C:\DeepLearning\DL_project\Utils\Preprocess.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using

[INFO] Train: (472432, 776)
[INFO] Val: (59054, 776)
[INFO] Test: (59054, 776)
Input dim: 776


In [4]:
# Load AutoEncoder checkpoint
checkpoint = torch.load("checkpoints/autoencoder.pt", map_location=DEVICE)

model = AutoEncoder(
    input_dim=checkpoint["input_dim"],
    latent_dim=checkpoint["latent_dim"],
    hidden_dims=checkpoint["hidden_dims"],
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Extract Features
X_train, y_train = extract_features(model, train_loader_full, DEVICE)
X_val, y_val = extract_features(model, val_loader, DEVICE)
X_test, y_test = extract_features(model, test_loader, DEVICE)
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

# Train distribution
print("Train distribution:")
n_pos = (y_train == 1).sum()
n_neg = (y_train == 0).sum()
scale_pos_weight = n_neg / n_pos

print("0:", n_neg)
print("1:", n_pos)
print("scale_pos_weight:", scale_pos_weight)

Shapes: (472432, 1037) (59054, 1037) (59054, 1037)
Train distribution:
0: 455902
1: 16530
scale_pos_weight: 27.580278281911674


In [5]:
# Parameters
param_grid = {
    "iterations": [2000, 3000],
    "depth": [8, 10],
    "learning_rate": [0.03]
}
weight_modes = ["balanced", "weighted"]

# Train CatBoost
def train_and_eval(params, weight_mode):
    if weight_mode == "balanced":
        weight_params = {"auto_class_weights": "Balanced"}
    else:
        weight_params = {"scale_pos_weight": scale_pos_weight}
        
    model_cb = CatBoostClassifier(
        iterations=params["iterations"],
        depth=params["depth"],
        learning_rate=params["learning_rate"],
        eval_metric="AUC",
        loss_function="Logloss",
        **weight_params,
        l2_leaf_reg=10,
        random_strength=1,
        bagging_temperature=1,
        early_stopping_rounds=100,
        random_seed=42,
        verbose=100
    )
    model_cb.fit(X_train, y_train, eval_set=(X_val, y_val))

    # Validation probabilities
    y_proba_val = model_cb.predict_proba(X_val)[:, 1]

    # Best Threshold (only on validation)
    best_f2 = -1
    best_thresh = 0.5

    for t in np.arange(0.01, 0.99, 0.005):
        y_pred = (y_proba_val >= t).astype(int)
        f2 = fbeta_score(y_val, y_pred, beta=2)
        if f2 > best_f2:
            best_f2 = f2
            best_thresh = t
    return model_cb, best_f2, best_thresh

In [6]:
# Grid search
best_val_f2 = 0
best_model = None
best_params = None
best_threshold = None
best_weight_mode = None

for mode in weight_modes:
    print(f"\n===== Weight mode: {mode} =====")
    for it in param_grid["iterations"]:
        for d in param_grid["depth"]:
            for lr in param_grid["learning_rate"]:
                params = {
                    "iterations": it,
                    "depth": d,
                    "learning_rate": lr
                }
                print(f"Training: {params}")
                model_cb, f2, thresh = train_and_eval(params, mode)
                print(f"Validation F2: {f2:.4f} | threshold: {thresh:.3f}")
                if f2 > best_val_f2:
                    best_val_f2 = f2
                    best_model = model_cb
                    best_params = params
                    best_threshold = thresh
                    best_weight_mode = mode

# Save best model
best_model.save_model("checkpoints/best_catboost.cbm")
best_config = {
    "best_val_f2": float(best_val_f2),
    "best_threshold": float(best_threshold),
    "params": best_params,
    "weight_mode": best_weight_mode
}
with open("checkpoints/best_catboost_params.json", "w") as f:
    json.dump(best_config, f, indent=4)

print("Saved best catboost model + parameters")


===== Weight mode: balanced =====
Training: {'iterations': 2000, 'depth': 8, 'learning_rate': 0.03}
0:	test: 0.8442370	best: 0.8442370 (0)	total: 432ms	remaining: 14m 23s
100:	test: 0.8934131	best: 0.8934131 (100)	total: 26s	remaining: 8m 8s
200:	test: 0.9087184	best: 0.9087184 (200)	total: 50.9s	remaining: 7m 35s
300:	test: 0.9173252	best: 0.9173252 (300)	total: 1m 14s	remaining: 7m 2s
400:	test: 0.9236521	best: 0.9236521 (400)	total: 1m 40s	remaining: 6m 39s
500:	test: 0.9314131	best: 0.9314131 (500)	total: 2m 6s	remaining: 6m 19s
600:	test: 0.9375596	best: 0.9375596 (600)	total: 2m 33s	remaining: 5m 56s
700:	test: 0.9418233	best: 0.9418233 (700)	total: 2m 59s	remaining: 5m 33s
800:	test: 0.9448091	best: 0.9448091 (800)	total: 3m 26s	remaining: 5m 8s
900:	test: 0.9472034	best: 0.9472034 (900)	total: 3m 53s	remaining: 4m 44s
1000:	test: 0.9489498	best: 0.9489498 (1000)	total: 4m 19s	remaining: 4m 18s
1100:	test: 0.9501931	best: 0.9501931 (1100)	total: 4m 45s	remaining: 3m 53s
1200:	t

In [7]:
# Final Test Evaluation
print("\n=== TEST EVALUATION ===")

# Use best model
y_proba_test = best_model.predict_proba(X_test)[:, 1]

# Default threshold (0.5)
y_pred_default = (y_proba_test >= 0.5).astype(int)

print("=== DEFAULT THRESHOLD (0.5) ===")
print(classification_report(y_test, y_pred_default))
print("ROC_AUC:", roc_auc_score(y_test, y_proba_test))
print("PR_AUC:", average_precision_score(y_test, y_proba_test))
print("ACCURACY:", accuracy_score(y_test, y_pred_default))
print("PRECISION:", precision_score(y_test, y_pred_default))
print("RECALL:", recall_score(y_test, y_pred_default))
print("F1:", f1_score(y_test, y_pred_default))
print("F2:", fbeta_score(y_test, y_pred_default, beta=2))


=== TEST EVALUATION ===
=== DEFAULT THRESHOLD (0.5) ===
              precision    recall  f1-score   support

         0.0       0.99      0.98      0.99     56988
         1.0       0.59      0.76      0.67      2066

    accuracy                           0.97     59054
   macro avg       0.79      0.87      0.83     59054
weighted avg       0.98      0.97      0.97     59054

ROC_AUC: 0.957531131534901
PR_AUC: 0.7685646512761497
ACCURACY: 0.9734141633081587
PRECISION: 0.5938682816048448
RECALL: 0.7594385285575992
F1: 0.6665250637213254
F2: 0.7193288098294517


In [8]:
# Evaluation: Best Threshold (From Validation)
y_pred_final = (y_proba_test >= best_threshold).astype(int)

# Final evaluation
print("\n=== FINAL RESULTS (OPTIMISED THRESHOLD) ===")
print("Weight mode:", best_weight_mode)
print("Threshold:", best_threshold)
print("ACCURACY:", accuracy_score(y_test, y_pred_final))
print("PRECISION:", precision_score(y_test, y_pred_final))
print("RECALL:", recall_score(y_test, y_pred_final))
print("F1:", f1_score(y_test, y_pred_final))
print("F2:", fbeta_score(y_test, y_pred_final, beta=2))
print(classification_report(y_test, y_pred_final))

# Save threshold + metadata
checkpoint = {
    "model_type": "catboost",
    "best_threshold": float(best_threshold),
    "best_val_f2": float(best_val_f2),
    "weight_mode": best_weight_mode,
    "test_roc_auc": float(roc_auc_score(y_test, y_proba_test)),
    "test_pr_auc": float(average_precision_score(y_test, y_proba_test)),
    "input_dim": int(input_dim), # metadata
    "latent_dim": 256,
    "scale_pos_weight": float(scale_pos_weight),
    "features_dim": int(X_train.shape[1])
}

with open("checkpoints/catboost_data_checkpoint.json", "w") as f:
    json.dump(checkpoint, f, indent=4)

print("All checkpoints saved!")


=== FINAL RESULTS (OPTIMISED THRESHOLD) ===
Weight mode: balanced
Threshold: 0.57
ACCURACY: 0.9779015816032783
PRECISION: 0.6665207877461706
RECALL: 0.7371732817037754
F1: 0.7000689496667433
F2: 0.721869371504408
              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99     56988
         1.0       0.67      0.74      0.70      2066

    accuracy                           0.98     59054
   macro avg       0.83      0.86      0.84     59054
weighted avg       0.98      0.98      0.98     59054

All checkpoints saved!


In [6]:
# Get Validation PR-AUC
model = CatBoostClassifier()
model.load_model("checkpoints/best_catboost.cbm")

y_proba_val = model.predict_proba(X_val)[:, 1]
pr_auc = average_precision_score(y_val, y_proba_val)

print(f"Validation PR-AUC: {pr_auc:.4f}")

Validation PR-AUC: 0.7878
